# ResNet-18 — CIFAR-10: PC-ALM vs Backpropagation

Statistical comparison of PC-ALM (Augmented Lagrangian Predictive Coding)
and standard backpropagation on a full ResNet-18, using the paired-trial
A/B experiment harness (`ABExperiment`).

PC-ALM augments standard PC inference with per-layer Lagrange multipliers
(dual variables) that accumulate prediction errors across inference steps.
At convergence in linear networks, the duals recover exact backpropagation
gradients. In nonlinear networks, PC-ALM closes the PC-BP gap, especially
in deep narrow regimes where standard PC underperforms.

Reference: Seely & Gould, "Augmented Lagrangian Predictive Coding", arXiv:2605.31022

**Architecture** (CIFAR-10 variant — no 7x7 conv or maxpool, identical
topology for both arms):
```
input(32,32,3) -> stem(32,32,32, 3x3)
-> Stage 1: 2 residual blocks (32,32,32)
-> Stage 2: 2 residual blocks (16,16,64)
-> Stage 3: 2 residual blocks (8,8,128)
-> Stage 4: 2 residual blocks (4,4,256)
-> GlobalAvgPool -> Linear(10, softmax+CE)
```

Each residual block:
```
x -> conv_a(3x3, act) -> conv_b(3x3, act) -> skip(sum)
```

Skip connections use `SkipConnection` (same dims) or 1x1 conv (downsample).
~2.8M parameters.

**Arms** — identical topology, different training dynamics:

| Arm      | Init              | Activation | Training                           |
|----------|-------------------|------------|------------------------------------|
| ALM      | XavierInitializer | tanh*      | ALM inference (dual + primal)      |
| Backprop | XavierInitializer | relu*      | Single forward pass + autodiff     |

*Configurable via `alm_activation_name` / `bp_activation_name`:
`relu`, `tanh`, `gelu`, `leaky_relu`. tanh is recommended for ALM
(bounded, non-zero gradients everywhere); relu is standard for backprop.

Note: muPC scaling (used in the original standard-PC version of this
notebook) is intentionally removed for ALM — muPC's gradient scaling
was designed for standard PC dynamics and conflicts with ALM's dual
variable mechanism.

Includes cosine LR schedule with warmup and optional data augmentation
(random horizontal flip + random crop with padding).

**Reports:**
- Per-trial accuracy results table
- Mean +/- standard error for each method
- Paired t-test with p-value
- Cohen's d effect size
- Power analysis: estimated n_trials needed for significance

## Imports & Setup

In [1]:
import jax
import numpy as np
import optax
import time

from fabricpc.nodes import ConvNode, Linear, IdentityNode, SkipConnection, AvgPool
from fabricpc.core.topology import Edge
from fabricpc.graph_assembly import TaskMap, graph
from fabricpc.core.inference import InferenceALM
from fabricpc.graph_initialization import initialize_params
from fabricpc.core.activations import (
    IdentityActivation,
    ReLUActivation,
    TanhActivation,
    GeluActivation,
    LeakyReLUActivation,
    SoftmaxActivation,
)
from fabricpc.core.energy import CrossEntropyEnergy
from fabricpc.core.initializers import XavierInitializer
from fabricpc.training import train_pcn, evaluate_pcn
from fabricpc.training.train_backprop import train_backprop, evaluate_backprop
from fabricpc.experiments import ExperimentArm, ABExperiment
from fabricpc.utils.data.dataloader import Cifar10Loader
from fabricpc import setup_jax

setup_jax()
jax.config.update("jax_default_prng_impl", "threefry2x32")

/home/shamir/jax-cuda-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Helper Classes & Functions

In [2]:
def get_activation(name):
    factories = {
        "relu": ReLUActivation,
        "tanh": TanhActivation,
        "gelu": GeluActivation,
        "leaky_relu": lambda: LeakyReLUActivation(alpha=0.1),
    }
    if name not in factories:
        raise ValueError(f"Unknown activation: {name}. Choose from {list(factories)}")
    return factories[name]()


class AugmentedCifar10Loader:
    """Wraps Cifar10Loader with random horizontal flip and random crop+pad."""

    def __init__(self, base_loader, seed=42, pad=4):
        self.base_loader = base_loader
        self.seed = seed
        self.pad = pad
        self._epoch = 0

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self._epoch)
        self._epoch += 1
        pad = self.pad
        for images, labels in self.base_loader:
            flip_mask = rng.random(images.shape[0]) > 0.5
            images[flip_mask] = images[flip_mask, :, ::-1, :]

            padded = np.pad(
                images, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode="reflect"
            )
            B, H, W, C = images.shape
            crop_y = rng.integers(0, 2 * pad + 1, size=B)
            crop_x = rng.integers(0, 2 * pad + 1, size=B)
            for i in range(B):
                images[i] = padded[
                    i, crop_y[i] : crop_y[i] + H, crop_x[i] : crop_x[i] + W, :
                ]

            yield images, labels

    def __len__(self):
        return len(self.base_loader)

## ResNet-18 Graph Builder

In [3]:
def make_residual_block(
    prev_node,
    channels,
    stride,
    block_name,
    weight_init,
    activation=ReLUActivation(),
):
    """
    Create one residual block: conv_a -> conv_b(act) -> skip(sum).

    Activation is applied on the main path before summation. The skip path
    passes through without activation, preserving gradient flow.

    Returns:
        (nodes_list, edges_list, skip_node) where skip_node is the block output.
    """
    in_h, in_w, in_channels = prev_node._shape

    if stride == 1:
        out_h, out_w = in_h, in_w
    else:
        out_h, out_w = in_h // stride, in_w // stride

    nodes = []
    edges = []

    conv_a = ConvNode(
        shape=(out_h, out_w, channels),
        kernel_size=(3, 3),
        stride=(stride, stride),
        padding="SAME",
        activation=activation,
        weight_init=weight_init,
        name=f"{block_name}_conv_a",
    )

    conv_b = ConvNode(
        shape=(out_h, out_w, channels),
        kernel_size=(3, 3),
        stride=(1, 1),
        padding="SAME",
        activation=activation,
        weight_init=weight_init,
        name=f"{block_name}_conv_b",
    )

    skip_node = SkipConnection(
        shape=(out_h, out_w, channels),
        name=f"{block_name}_skip_sum",
    )

    nodes.extend([conv_a, conv_b, skip_node])

    # Main path edges
    edges.append(Edge(source=prev_node, target=conv_a.slot("in")))
    edges.append(Edge(source=conv_a, target=conv_b.slot("in")))
    edges.append(Edge(source=conv_b, target=skip_node.slot("in")))

    # Skip connection: the stream enters the merge's unscaled "skip" slot
    # (via a 1x1 projection when the block downsamples).
    needs_downsample = (stride != 1) or (in_channels != channels)
    if needs_downsample:
        conv_skip = ConvNode(
            shape=(out_h, out_w, channels),
            kernel_size=(1, 1),
            stride=(stride, stride),
            padding="SAME",
            activation=IdentityActivation(),
            weight_init=weight_init,
            name=f"{block_name}_skip",
        )
        nodes.append(conv_skip)
        edges.append(Edge(source=prev_node, target=conv_skip.slot("in")))
        edges.append(Edge(source=conv_skip, target=skip_node.slot("skip")))
    else:
        edges.append(Edge(source=prev_node, target=skip_node.slot("skip")))

    return nodes, edges, skip_node


def build_resnet18(
    weight_init,
    scaling=None,
    output_weight_init=XavierInitializer(),
    activation=ReLUActivation(),
    *,
    infer_steps,
    eta_infer,
    alpha=1.0,
    rho=1.0,
    weight_credit_timing="pre_dual_energy",
):
    """
    Build ResNet-18 for CIFAR-10 as a predictive coding graph.

    The same builder serves both arms: graph() defaults
    graph_state_initializer to FeedforwardStateInit, which backprop
    training requires. muPC scaling (optional) is only used by the ALM arm.

    Args:
        weight_init: InitializerBase for conv/linear weights.
        scaling: Optional MuPCConfig for muPC parameterization.
        output_weight_init: InitializerBase for the output layer
            (default: XavierInitializer).
        activation: Activation for hidden conv layers (default: ReLU).
        infer_steps: Number of ALM inference steps.
        eta_infer: Inference rate.
        alpha: ALM dual step size (default: 1.0).
        rho: ALM penalty strength (default: 1.0).
        weight_credit_timing: When to snapshot duals for weight gradients.
            "pre_dual_energy" (default) — duals from the last primal step.
            "post_dual_energy" — one extra dual update after the final primal.

    Returns:
        GraphStructure ready for initialize_params().
    """
    # Input
    input_node = IdentityNode(shape=(32, 32, 3), name="input")

    # Stem convolution: 3x3, 32 channels, no maxpool (CIFAR is 32x32)
    stem = ConvNode(
        shape=(32, 32, 32),
        kernel_size=(3, 3),
        stride=(1, 1),
        padding="SAME",
        activation=activation,
        weight_init=weight_init,
        name="stem",
    )

    all_nodes = [input_node, stem]
    all_edges = [Edge(source=input_node, target=stem.slot("in"))]

    # Build 4 stages with [2, 2, 2, 2] blocks
    stage_configs = [
        (32, 1, 2),   # (channels, first_stride, num_blocks)
        (64, 2, 2),
        (128, 2, 2),
        (256, 2, 2),
    ]

    prev = stem
    for stage_idx, (channels, first_stride, num_blocks) in enumerate(stage_configs, 1):
        for block_idx in range(num_blocks):
            stride = first_stride if block_idx == 0 else 1
            block_name = f"s{stage_idx}b{block_idx + 1}"

            nodes, edges, add_node = make_residual_block(
                prev_node=prev,
                channels=channels,
                stride=stride,
                block_name=block_name,
                weight_init=weight_init,
                activation=activation,
            )
            all_nodes.extend(nodes)
            all_edges.extend(edges)
            prev = add_node

    # Global average pooling: (B, 4, 4, 256) -> (B, 256)
    avg_pool = AvgPool(shape=(256,), name="avgpool", global_pool=True)
    all_nodes.append(avg_pool)
    all_edges.append(Edge(source=prev, target=avg_pool.slot("in")))

    # Output: Linear(10) with softmax + cross-entropy
    output = Linear(
        shape=(10,),
        activation=SoftmaxActivation(),
        energy=CrossEntropyEnergy(),
        flatten_input=True,
        weight_init=output_weight_init,
        name="output",
    )
    all_nodes.append(output)
    all_edges.append(Edge(source=avg_pool, target=output.slot("in")))

    # Build graph
    structure = graph(
        nodes=all_nodes,
        edges=all_edges,
        task_map=TaskMap(x=input_node, y=output),
        inference=InferenceALM(
            eta_infer=eta_infer, infer_steps=infer_steps,
            alpha=alpha, rho=rho,
            weight_credit_timing=weight_credit_timing,
        ),
        scaling=scaling,
    )

    return structure

## Configuration

In [4]:
# --- Experiment settings ---
n_trials = 3             # independent paired trials per method (>=2 for t-test)
verbose = False          # True prints per-epoch loss for each trial

# --- Training hyperparameters ---
# ResNet-18 ALM is heavy: ~6-8 min/epoch on an RTX3090 at these settings.
# num_epochs=2 is a smoke test; use 30+ for a meaningful comparison.
num_epochs = 30
batch_size = 128         # reduced from 256 to ease GPU memory pressure (OOM)
lr = 0.001
weight_decay = 0.01
augment = False          # random crop+pad and horizontal flip

# --- Activations (per arm) ---
alm_activation_name = "tanh"   # bounded, smooth — best for iterative inference
bp_activation_name  = "relu"   # standard for end-to-end backprop

# --- ALM inference settings ---
# 200 steps with conservative eta gives ALM more room to converge through
# ResNet-18's 8 residual blocks (~18 effective layers). Lower alpha slows
# dual accumulation to avoid overshooting in deep networks.
infer_steps = 200        # increased from 120 — deeper nets need more steps
eta_infer   = 0.05       # halved from 0.1 — more conservative primal updates
alm_alpha   = 0.5        # halved from 1.0 — gentler dual accumulation
alm_rho     = 1.0        # penalty strength — should match energy precision

# --- Weight credit timing ---
# Controls when duals are snapshotted for weight gradients:
#   "pre_dual_energy"  — use duals from the last primal step (default, matches reference)
#   "post_dual_energy" — one extra dual update after the final primal step
weight_credit_timing = "pre_dual_energy"

train_config = {"num_epochs": num_epochs}

## Model Factories

In [5]:
def create_alm_model(rng_key):
    """ALM arm — Xavier init, no muPC (muPC was designed for standard PC
    and its gradient scaling conflicts with ALM's dual variable mechanism)."""
    structure = build_resnet18(
        weight_init=XavierInitializer(),
        activation=get_activation(alm_activation_name),
        infer_steps=infer_steps,
        eta_infer=eta_infer,
        alpha=alm_alpha,
        rho=alm_rho,
        weight_credit_timing=weight_credit_timing,
    )
    params = initialize_params(structure, rng_key)
    return params, structure


def create_backprop_model(rng_key):
    """Backprop arm — standard Xavier init, no muPC scaling."""
    structure = build_resnet18(
        weight_init=XavierInitializer(),
        activation=get_activation(bp_activation_name),
        infer_steps=infer_steps,
        eta_infer=eta_infer,
        alpha=alm_alpha,
        rho=alm_rho,
        weight_credit_timing=weight_credit_timing,
    )
    params = initialize_params(structure, rng_key)
    return params, structure


# Sanity check: build once to report graph size / parameter count
_probe_params, _probe_structure = create_backprop_model(jax.random.PRNGKey(0))
print(f"Model: {len(_probe_structure.nodes)} nodes, {len(_probe_structure.edges)} edges")
total_params = sum(p.size for p in jax.tree_util.tree_leaves(_probe_params))
print(f"Total parameters: {total_params:,}")
del _probe_params, _probe_structure

Model: 31 nodes, 38 edges
Total parameters: 2,795,210


## Run Experiment

In [6]:
print("=" * 70)
print("ResNet-18 on CIFAR-10: PC-ALM vs Backpropagation")
print("=" * 70)
print(
    "Architecture: stem(32) -> s1[2x32] -> s2[2x64] -> s3[2x128] "
    "-> s4[2x256] -> avgpool -> 10"
)
print(
    f"ALM: {alm_activation_name} + Xavier, {infer_steps} inference steps "
    f"@ eta={eta_infer}, alpha={alm_alpha}, rho={alm_rho}  |  "
    f"Backprop: {bp_activation_name} + Xavier"
)
print(f"Weight credit timing: {weight_credit_timing}")
print(
    f"Epochs: {num_epochs}  |  Batch size: {batch_size}  |  LR: {lr}  "
    f"|  Augment: {augment}"
)
print(f"Trials: {n_trials}")
print()

# Cosine LR schedule with warmup (shared by both arms)
steps_per_epoch = len(
    Cifar10Loader("train", batch_size=batch_size, shuffle=True, seed=0)
)
total_steps = num_epochs * steps_per_epoch
warmup_steps = int(0.05 * total_steps)

schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=lr,
    warmup_steps=warmup_steps,
    decay_steps=total_steps,
    end_value=lr * 0.01,
)
optimizer = optax.adamw(schedule, weight_decay=weight_decay)

arm_alm = ExperimentArm(
    name="ALM",
    model_factory=create_alm_model,
    train_fn=train_pcn,
    eval_fn=evaluate_pcn,
    optimizer=optimizer,
    train_config=train_config,
)

arm_bp = ExperimentArm(
    name="Backprop",
    model_factory=create_backprop_model,
    train_fn=train_backprop,
    eval_fn=evaluate_backprop,
    optimizer=optimizer,
    train_config=train_config,
)


def data_loader_factory(seed):
    base_train = Cifar10Loader(
        "train", batch_size=batch_size, shuffle=True, seed=seed
    )
    train_loader = (
        AugmentedCifar10Loader(base_train, seed=seed) if augment else base_train
    )
    test_loader = Cifar10Loader("test", batch_size=batch_size, shuffle=False)
    return train_loader, test_loader


experiment = ABExperiment(
    arm_a=arm_alm,
    arm_b=arm_bp,
    metric="accuracy",
    data_loader_factory=data_loader_factory,
    n_trials=n_trials,
    verbose=verbose,
)

results = experiment.run()
results.print_summary()

ResNet-18 on CIFAR-10: PC-ALM vs Backpropagation
Architecture: stem(32) -> s1[2x32] -> s2[2x64] -> s3[2x128] -> s4[2x256] -> avgpool -> 10
ALM: tanh + Xavier, 200 inference steps @ eta=0.05, alpha=0.5, rho=1.0  |  Backprop: relu + Xavier
Weight credit timing: pre_dual_energy
Epochs: 30  |  Batch size: 128  |  LR: 0.001  |  Augment: False
Trials: 3

--- Trial 1/3 (seed=0) ---


Epoch 30/30: 100%|██████████| 11730/11730 [3:40:46<00:00,  1.13s/it, energy=0.0342, epoch=30/30] 


  ALM: accuracy=0.5704  (train: 13247.0s)
  Backprop: accuracy=0.8042  (train: 242.1s)
--- Trial 2/3 (seed=1000) ---


Epoch 30/30: 100%|██████████| 11730/11730 [3:39:31<00:00,  1.12s/it, energy=0.0419, epoch=30/30] 


  ALM: accuracy=0.5815  (train: 13171.2s)
  Backprop: accuracy=0.8069  (train: 230.1s)
--- Trial 3/3 (seed=2000) ---


Epoch 5/30:  17%|█▋        | 1940/11730 [36:39<3:07:33,  1.15s/it, energy=2.0627, epoch=5/30]

KeyboardInterrupt: 